# Task 1 (Player Style) — Trích xuất profile cầu thủ per-90 từ Premier League + La Liga + Serie A (2015/2016)

**Mục tiêu:** khác với Task 1 trước (đơn vị phân tích = 1 shot), notebook này có đơn vị phân tích = **1 cầu thủ** (gộp cả mùa giải), phục vụ bài toán **Phân cụm phong cách/vị trí thi đấu (Player Style Clustering)**.

**Phạm vi dữ liệu:** cố định 3 giải hàng đầu châu Âu, cùng mùa **2015/2016** — mùa duy nhất cả 3 giải đều có **full 380 trận** trong StatsBomb open-data:

| Giải | `competition_id` | `season_id` |
|---|---|---|
| Premier League | 2 | 27 |
| La Liga | 11 | 27 |
| Serie A | 12 | 27 |

**Nguồn dữ liệu:** Kaggle Dataset `saurabhshahane/statsbomb-football-data` (mount tại `/kaggle/input/statsbomb-football-data/`), dùng cơ chế **auto-discovery** giống notebook World Cup trước — tự quét thư mục, không phụ thuộc cấu trúc nesting cụ thể.

**2 bước tính toán quan trọng nhất trong notebook này (khác hẳn Task 1 Shot Zone):**
1. **Tính số phút thi đấu của từng cầu thủ** — từ `Starting XI`, `Substitution`, và **thẻ đỏ/thẻ vàng thứ 2** (dễ bị bỏ sót nếu chỉ nhìn Substitution) — dùng làm mẫu số cho mọi chỉ số per-90.
2. **Tổng hợp nhiều loại event** (Pass, Shot, Dribble, Duel, Interception, Pressure, Clearance, Foul...) theo từng cầu thủ, cộng dồn qua toàn bộ trận đã đá trong mùa.


## 1.0 — Import & Config

In [1]:
import json
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [2]:
# ---------------- Config ----------------

COMPETITIONS = [
    (2, 27, "Premier League"),
    (11, 27, "La Liga"),
    (12, 27, "Serie A"),
]

MIN_MINUTES = 900  # ngưỡng lọc: chỉ giữ cầu thủ có >= 900 phút thi đấu (~10 trận đầy đủ) trong TOÀN BỘ 3 giải

MAX_MATCHES_PER_COMPETITION = None  # đổi thành số nhỏ (vd. 5) để test nhanh trước khi chạy full 380x3 trận

KAGGLE_INPUT_DIR = Path("/kaggle/input/datasets/saurabhshahane/statsbomb-football-data/data")
WORK_DIR = Path("/kaggle/working")
OUTPUT_CSV = WORK_DIR / "player_style_features_pl_laliga_seriea_1516.csv"


## 1.1 — Auto-discovery cấu trúc thư mục dataset (giống notebook World Cup)

In [3]:
def discover_paths(base_dir: Path):
    competitions_path = None
    matches_index = {}
    events_index = {}

    all_json_files = list(base_dir.rglob("*.json"))

    for p in all_json_files:
        parts = p.parts
        if p.name == "competitions.json" and competitions_path is None:
            competitions_path = p
            continue
        if len(parts) >= 3 and parts[-3].lower() == "matches":
            competition_id, season_id = parts[-2], p.stem
            if competition_id.isdigit() and season_id.isdigit():
                matches_index[(int(competition_id), int(season_id))] = p
                continue
        if len(parts) >= 2 and parts[-2].lower() == "events":
            match_id = p.stem
            if match_id.isdigit():
                events_index[match_id] = p

    return competitions_path, matches_index, events_index, len(all_json_files)


assert KAGGLE_INPUT_DIR.exists(), (
    f"Không tìm thấy thư mục {KAGGLE_INPUT_DIR}. Kiểm tra lại đã add đúng dataset chưa."
)

competitions_path, matches_index, events_index, n_json_files = discover_paths(KAGGLE_INPUT_DIR)
print(f"Tổng số file .json quét được: {n_json_files}")
print(f"competitions.json: {competitions_path}")
print(f"Số entry matches đã index: {len(matches_index)}")
print(f"Số entry events đã index : {len(events_index)}")

for comp_id, season_id, label in COMPETITIONS:
    status = "OK" if (comp_id, season_id) in matches_index else "KHÔNG TÌM THẤY"
    print(f"{label}: competition_id={comp_id}, season_id={season_id} -> {status}")


Tổng số file .json quét được: 8977
competitions.json: /kaggle/input/datasets/saurabhshahane/statsbomb-football-data/data/competitions.json
Số entry matches đã index: 80
Số entry events đã index : 4235
Premier League: competition_id=2, season_id=27 -> OK
La Liga: competition_id=11, season_id=27 -> OK
Serie A: competition_id=12, season_id=27 -> OK


In [4]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_match_ids(competition_id: int, season_id: int):
    matches_path = matches_index.get((competition_id, season_id))
    if matches_path is None:
        raise FileNotFoundError(f"Không tìm thấy matches cho ({competition_id},{season_id})")
    matches = load_json(matches_path)
    return [m["match_id"] for m in matches]


def get_events(match_id: int):
    events_path = events_index.get(str(match_id))
    if events_path is None:
        raise FileNotFoundError(f"Không tìm thấy events cho match_id={match_id}")
    return load_json(events_path)


## 1.2 — Hàm tính số phút thi đấu của từng cầu thủ trong 1 trận

Xử lý 3 nguồn kết thúc thời gian thi đấu của 1 cầu thủ: `Substitution` (bị thay ra), thẻ đỏ trực tiếp hoặc thẻ vàng thứ 2 (`Foul Committed`/`Bad Behaviour` có `card.name` là "Red Card"/"Second Yellow"). Nếu cầu thủ không có sự kiện nào trong 3 loại trên, coi như đá đến hết trận (`match_end_minute`, lấy từ event `Half End`).

**Giới hạn đã biết (chấp nhận được):** sự kiện `Player Off` không kèm `Substitution` (thường là chấn thương tạm thời, cầu thủ quay lại ngay sau đó qua `Player On`) được bỏ qua — vì phần lớn là tạm thời, ảnh hưởng không đáng kể đến thống kê per-90 khi cộng dồn cả mùa.

In [5]:
RED_CARD_NAMES = {"Red Card", "Second Yellow"}


def compute_match_minutes(events):
    half_end_minutes = {}
    for e in events:
        if e["type"]["name"] == "Half End":
            p = e["period"]
            half_end_minutes[p] = max(half_end_minutes.get(p, 0), e["minute"])
    valid_periods = [p for p in half_end_minutes if p in (1, 2, 3, 4)]
    match_end_minute = max(half_end_minutes[p] for p in valid_periods) if valid_periods else 90

    start_minute = {}
    exit_minute = {}
    player_name = {}
    player_team = {}

    def register_exit(pid, minute):
        if pid not in exit_minute or minute < exit_minute[pid]:
            exit_minute[pid] = minute

    for e in events:
        etype = e["type"]["name"]

        if etype == "Starting XI":
            team = e["team"]["name"]
            for p in e.get("tactics", {}).get("lineup", []):
                pid = p["player"]["id"]
                start_minute[pid] = 0
                player_name[pid] = p["player"]["name"]
                player_team[pid] = team

        elif etype == "Substitution":
            off_id = e["player"]["id"]
            minute = e["minute"]
            register_exit(off_id, minute)
            replacement = e.get("substitution", {}).get("replacement")
            if replacement:
                on_id = replacement["id"]
                start_minute[on_id] = minute
                player_name[on_id] = replacement["name"]
                player_team[on_id] = e["team"]["name"]

        elif etype == "Foul Committed":
            card = e.get("foul_committed", {}).get("card", {}).get("name")
            if card in RED_CARD_NAMES:
                register_exit(e["player"]["id"], e["minute"])

        elif etype == "Bad Behaviour":
            card = e.get("bad_behaviour", {}).get("card", {}).get("name")
            if card in RED_CARD_NAMES:
                register_exit(e["player"]["id"], e["minute"])

    minutes_played = {}
    for pid, s in start_minute.items():
        end_ = exit_minute.get(pid, match_end_minute)
        minutes_played[pid] = max(0, end_ - s)

    return minutes_played, player_name, player_team


## 1.3 — Hàm tổng hợp feature theo cầu thủ từ mọi loại event trong 1 trận

`TOUCH_EVENT_TYPES` dùng để tính touches / vị trí trung bình / touches trong vòng cấm — chỉ tính trên các loại event thể hiện cầu thủ đang trực tiếp xử lý bóng (không tính `Pressure` vì đó là hành động lên đối phương, không phải chạm bóng).

In [6]:
TOUCH_EVENT_TYPES = {
    "Pass", "Shot", "Dribble", "Carry", "Ball Receipt*",
    "Miscontrol", "Clearance", "Interception", "Ball Recovery", "Goal Keeper",
}
TACKLE_SUCCESS_OUTCOMES = {"Won", "Success", "Success In Play", "Success Out"}
INTERCEPTION_SUCCESS_OUTCOMES = {"Won", "Success", "Success In Play", "Success Out"}


def scan_aerial_won(event):
    # Quét mọi sub-object lồng trong event để tìm cờ aerial_won == True,
    # bất kể nằm trong loại event nào (Shot, Clearance, Duel, Miscontrol...)
    for v in event.values():
        if isinstance(v, dict) and v.get("aerial_won") is True:
            return True
    return False


def new_stat_row():
    return defaultdict(float)


def aggregate_match_events(events):
    stats = defaultdict(new_stat_row)
    positions = defaultdict(Counter)

    for e in events:
        player = e.get("player")
        if not player:
            continue
        pid = player["id"]
        etype = e["type"]["name"]
        row = stats[pid]

        pos = e.get("position", {}).get("name")
        if pos:
            positions[pid][pos] += 1

        loc = e.get("location")
        if etype in TOUCH_EVENT_TYPES and loc:
            x, y = loc[0], loc[1]
            row["touches"] += 1
            row["sum_x"] += x
            row["sum_y"] += y
            row["sum_x2"] += x * x
            row["sum_y2"] += y * y
            if x >= 102 and 18 <= y <= 62:
                row["touches_in_box"] += 1

        if scan_aerial_won(e):
            row["aerial_won"] += 1

        if etype == "Shot":
            shot = e.get("shot", {})
            row["shots"] += 1
            if shot.get("outcome", {}).get("name") == "Goal" and shot.get("type", {}).get("name") != "Penalty":
                row["np_goals"] += 1
            if loc:
                dx, dy = 120 - loc[0], 40 - loc[1]
                row["sum_shot_distance"] += (dx ** 2 + dy ** 2) ** 0.5
                row["n_shots_with_distance"] += 1

        elif etype == "Dribble":
            row["dribbles"] += 1
            if e.get("dribble", {}).get("outcome", {}).get("name") == "Complete":
                row["dribbles_complete"] += 1

        elif etype == "Pass":
            passv = e.get("pass", {})
            row["passes"] += 1
            if "outcome" not in passv:
                row["passes_complete"] += 1
            if passv.get("shot_assist"):
                row["key_passes"] += 1
            if passv.get("goal_assist"):
                row["assists"] += 1
            if passv.get("cross"):
                row["crosses"] += 1
            if passv.get("technique", {}).get("name") == "Through Ball":
                row["through_balls"] += 1
            length = passv.get("length")
            if length is not None:
                row["sum_pass_length"] += length
                row["n_passes_with_length"] += 1
                if length > 30:
                    row["long_balls"] += 1

        elif etype == "Pressure":
            row["pressures"] += 1

        elif etype == "Duel":
            duel = e.get("duel", {})
            if duel.get("type", {}).get("name") == "Tackle":
                row["tackles"] += 1
                if duel.get("outcome", {}).get("name") in TACKLE_SUCCESS_OUTCOMES:
                    row["tackles_won"] += 1

        elif etype == "Interception":
            row["interceptions"] += 1
            outcome = e.get("interception", {}).get("outcome", {}).get("name")
            if outcome in INTERCEPTION_SUCCESS_OUTCOMES:
                row["interceptions_won"] += 1

        elif etype == "Ball Recovery":
            row["ball_recoveries"] += 1

        elif etype == "Clearance":
            row["clearances"] += 1

        elif etype == "Foul Committed":
            row["fouls_committed"] += 1

        elif etype == "Foul Won":
            row["fouls_won"] += 1

    return stats, positions


## 1.4 — Chạy trích xuất & tổng hợp toàn bộ 3 giải

In [7]:
global_minutes = defaultdict(float)
global_matches_played = defaultdict(int)
global_stats = defaultdict(new_stat_row)
global_positions = defaultdict(Counter)
global_names = {}
global_teams = defaultdict(Counter)
global_competitions = defaultdict(set)

failed_matches = []

for comp_id, season_id, label in COMPETITIONS:
    match_ids = get_match_ids(comp_id, season_id)
    if MAX_MATCHES_PER_COMPETITION is not None:
        match_ids = match_ids[:MAX_MATCHES_PER_COMPETITION]

    print(f"{label}: {len(match_ids)} trận sẽ được xử lý")

    for match_id in tqdm(match_ids, desc=label):
        try:
            events = get_events(match_id)

            minutes, names, teams = compute_match_minutes(events)
            stats, positions = aggregate_match_events(events)

            for pid, m in minutes.items():
                if m <= 0:
                    continue
                global_minutes[pid] += m
                global_matches_played[pid] += 1
                global_names[pid] = names[pid]
                global_teams[pid][teams[pid]] += 1
                global_competitions[pid].add(label)

            for pid, row in stats.items():
                g_row = global_stats[pid]
                for k, v in row.items():
                    g_row[k] += v

            for pid, counter in positions.items():
                global_positions[pid].update(counter)

        except Exception as ex:
            print(f"[WARN] Lỗi ở match_id={match_id} ({label}): {ex}")
            failed_matches.append((label, match_id))

print(f"\nTổng số cầu thủ (đã có ít nhất 1 phút thi đấu): {len(global_minutes)}")
if failed_matches:
    print(f"Số trận lỗi (bỏ qua): {len(failed_matches)}")


Premier League: 380 trận sẽ được xử lý


Premier League:   0%|          | 0/380 [00:00<?, ?it/s]

La Liga: 380 trận sẽ được xử lý


La Liga:   0%|          | 0/380 [00:00<?, ?it/s]

Serie A: 380 trận sẽ được xử lý


Serie A:   0%|          | 0/380 [00:00<?, ?it/s]


Tổng số cầu thủ (đã có ít nhất 1 phút thi đấu): 1622


## 1.5 — Gộp thành bảng, tính per-90 & lọc theo ngưỡng phút thi đấu

In [8]:
rows = []
for pid, minutes in global_minutes.items():
    row = global_stats[pid]
    factor = 90.0 / minutes if minutes > 0 else np.nan

    def rate(key):
        return row.get(key, 0.0) * factor

    def ratio(numer_key, denom_key):
        denom = row.get(denom_key, 0.0)
        return row.get(numer_key, 0.0) / denom if denom > 0 else np.nan

    def avg(sum_key, count_key):
        cnt = row.get(count_key, 0.0)
        return row.get(sum_key, 0.0) / cnt if cnt > 0 else np.nan

    touches = row.get("touches", 0.0)
    avg_x = row["sum_x"] / touches if touches > 0 else np.nan
    avg_y = row["sum_y"] / touches if touches > 0 else np.nan
    std_x = np.sqrt(max(row["sum_x2"] / touches - avg_x ** 2, 0)) if touches > 1 else np.nan
    std_y = np.sqrt(max(row["sum_y2"] / touches - avg_y ** 2, 0)) if touches > 1 else np.nan

    primary_team = global_teams[pid].most_common(1)[0][0] if global_teams[pid] else None
    primary_position = global_positions[pid].most_common(1)[0][0] if global_positions[pid] else None

    rows.append({
        "player_id": pid,
        "player_name": global_names.get(pid),
        "primary_team": primary_team,
        "competitions_played": ", ".join(sorted(global_competitions[pid])),
        "n_matches_played": global_matches_played[pid],
        "total_minutes_played": minutes,

        # --- Feature X (per-90 & tỷ lệ) ---
        "shots_p90": rate("shots"),
        "np_goals_p90": rate("np_goals"),
        "avg_shot_distance": avg("sum_shot_distance", "n_shots_with_distance"),
        "dribbles_p90": rate("dribbles"),
        "dribble_success_rate": ratio("dribbles_complete", "dribbles"),
        "touches_p90": rate("touches"),
        "touches_in_box_p90": rate("touches_in_box"),
        "passes_p90": rate("passes"),
        "pass_completion_pct": ratio("passes_complete", "passes"),
        "key_passes_p90": rate("key_passes"),
        "assists_p90": rate("assists"),
        "crosses_p90": rate("crosses"),
        "through_balls_p90": rate("through_balls"),
        "long_balls_p90": rate("long_balls"),
        "avg_pass_length": avg("sum_pass_length", "n_passes_with_length"),
        "pressures_p90": rate("pressures"),
        "tackles_p90": rate("tackles"),
        "tackle_success_rate": ratio("tackles_won", "tackles"),
        "interceptions_p90": rate("interceptions"),
        "interception_success_rate": ratio("interceptions_won", "interceptions"),
        "ball_recoveries_p90": rate("ball_recoveries"),
        "clearances_p90": rate("clearances"),
        "aerial_won_p90": rate("aerial_won"),
        "fouls_committed_p90": rate("fouls_committed"),
        "fouls_won_p90": rate("fouls_won"),
        "avg_location_x": avg_x,
        "avg_location_y": avg_y,
        "std_location_x": std_x,
        "std_location_y": std_y,

        # --- Field giấu để validate cụm sau này (KHÔNG dùng làm feature) ---
        "primary_position": primary_position,
    })

player_df = pd.DataFrame(rows)
print("Tổng số cầu thủ trước khi lọc:", len(player_df))

player_df_filtered = player_df[player_df["total_minutes_played"] >= MIN_MINUTES].reset_index(drop=True)
print(f"Số cầu thủ sau khi lọc (>= {MIN_MINUTES} phút): {len(player_df_filtered)}")


Tổng số cầu thủ trước khi lọc: 1622
Số cầu thủ sau khi lọc (>= 900 phút): 1016


## 1.6 — Kiểm tra nhanh trước khi lưu

In [9]:
print("Shape:", player_df_filtered.shape)
player_df_filtered.head(3)


Shape: (1016, 36)


,player_id,player_name,primary_team,competitions_played,n_matches_played,total_minutes_played,shots_p90,np_goals_p90,avg_shot_distance,dribbles_p90,...,ball_recoveries_p90,clearances_p90,aerial_won_p90,fouls_committed_p90,fouls_won_p90,avg_location_x,avg_location_y,std_location_x,std_location_y,primary_position
0,3339,Asmir Begović,Chelsea,Premier League,17,1472.0,0.000000,0.000000,NaN,0.000000,...,3.362772,0.000000,0.000000,0.000000,0.244565,9.588953,40.169961,6.030542,7.832626,Goalkeeper
1,5594,Branislav Ivanović,Chelsea,Premier League,33,3057.0,0.824338,0.058881,18.956525,0.529931,...,3.562316,3.856722,2.796860,1.148184,0.706575,60.060169,59.562310,27.428947,20.427508,Right Back
2,3456,Kurt Happy Zouma,Chelsea,Premier League,23,2013.0,0.715350,0.044709,16.912380,0.357675,...,2.369598,7.555887,3.979136,0.447094,0.402385,41.311802,52.834272,21.911993,12.956491,Right Center Back


In [10]:
missing = player_df_filtered.isna().sum()
missing[missing > 0].sort_values(ascending=False)


avg_shot_distance            75
interception_success_rate    72
tackle_success_rate          70
dribble_success_rate         42
dtype: int64

In [11]:
print(player_df_filtered["primary_position"].value_counts())
print()
print(player_df_filtered["competitions_played"].value_counts())


primary_position
Right Center Back            94
Right Back                   91
Left Back                    88
Left Center Back             83
Center Forward               80
Goalkeeper                   73
Left Wing                    61
Right Wing                   60
Right Defensive Midfield     49
Right Center Midfield        48
Left Defensive Midfield      46
Center Attacking Midfield    42
Left Center Midfield         41
Center Defensive Midfield    39
Right Center Forward         27
Left Midfield                25
Right Midfield               22
Left Center Forward          19
Right Wing Back              11
Left Wing Back                9
Center Back                   7
Right Attacking Midfield      1
Name: count, dtype: int64

competitions_played
Serie A                    339
La Liga                    338
Premier League             327
La Liga, Serie A             6
Premier League, Serie A      5
La Liga, Premier League      1
Name: count, dtype: int64


In [12]:
# Sanity check: tổng phút của 1 đội trong 1 trận luôn <= 22 người x số phút trận
# -> ở đây kiểm tra nhanh: không có ai có minutes_played âm hoặc vượt quá tổng số phút khả dĩ trong mùa
max_possible_minutes = 38 * 100  # ước lượng an toàn: tối đa 38 trận/giải, mỗi trận tối đa ~100 phút (kể cả bù giờ)
print("Có cầu thủ nào vượt ngưỡng bất thường không:", (player_df_filtered["total_minutes_played"] > max_possible_minutes).any())
print("Có cầu thủ nào âm phút không:", (player_df_filtered["total_minutes_played"] < 0).any())


Có cầu thủ nào vượt ngưỡng bất thường không: False
Có cầu thủ nào âm phút không: False


## 1.7 — Ghi ra file CSV

In [13]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
player_df_filtered.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {OUTPUT_CSV}  ({player_df_filtered.shape[0]} dòng, {player_df_filtered.shape[1]} cột)")


Đã lưu: /kaggle/working/player_style_features_pl_laliga_seriea_1516.csv  (1016 dòng, 36 cột)


In [14]:
check_df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
check_df.head(5)


,player_id,player_name,primary_team,competitions_played,n_matches_played,total_minutes_played,shots_p90,np_goals_p90,avg_shot_distance,dribbles_p90,...,ball_recoveries_p90,clearances_p90,aerial_won_p90,fouls_committed_p90,fouls_won_p90,avg_location_x,avg_location_y,std_location_x,std_location_y,primary_position
0,3339,Asmir Begović,Chelsea,Premier League,17,1472.0,0.000000,0.000000,NaN,0.000000,...,3.362772,0.000000,0.000000,0.000000,0.244565,9.588953,40.169961,6.030542,7.832626,Goalkeeper
1,5594,Branislav Ivanović,Chelsea,Premier League,33,3057.0,0.824338,0.058881,18.956525,0.529931,...,3.562316,3.856722,2.796860,1.148184,0.706575,60.060169,59.562310,27.428947,20.427508,Right Back
2,3456,Kurt Happy Zouma,Chelsea,Premier League,23,2013.0,0.715350,0.044709,16.912380,0.357675,...,2.369598,7.555887,3.979136,0.447094,0.402385,41.311802,52.834272,21.911993,12.956491,Right Center Back
3,3645,Gary Cahill,Chelsea,Premier League,23,2061.0,0.611354,0.087336,12.114271,0.174672,...,3.580786,6.550218,4.017467,1.048035,0.873362,41.635940,44.846527,22.276410,17.782090,Right Center Back
4,3957,César Azpilicueta Tanco,Chelsea,Premier League,37,3351.0,0.295434,0.053715,17.036638,0.698299,...,3.410922,3.303491,1.933751,1.289167,0.590868,61.356087,28.466955,26.820258,28.882777,Left Back


## Kết quả Task 1 (Player Style)

File `player_style_features_pl_laliga_seriea_1516.csv` — mỗi dòng là 1 cầu thủ đã đá >= 900 phút trong mùa 2015/2016 (gộp cả 3 giải), gồm:
- ID: `player_id`, `player_name`, `primary_team`, `competitions_played`, `n_matches_played`, `total_minutes_played`
- ~26 feature per-90/tỷ lệ (X cho Task 3-5 sau này)
- `primary_position`: vị trí thi đấu phổ biến nhất — **giữ riêng để validate cụm**, không đưa vào X

Bước tiếp theo (Task 2/3 — EDA) sẽ khám phá phân phối các feature này, kiểm tra tương quan/dư thừa thông tin, trước khi chuẩn hóa và đưa vào K-Means để tìm cụm phong cách/vị trí.
